In [1]:
import cv2
import numpy as np

In [2]:
def create_color_painting(image, d_size, sigma_color, sigma_space, repetition_count, downsample_factor=0.5):
  
    
    original_height, original_width = image.shape[:2]
    
    
    small_img = cv2.resize(image, (0, 0), fx=downsample_factor, fy=downsample_factor, interpolation=cv2.INTER_LINEAR)
    
    
    smoothed_img = small_img
    for _ in range(repetition_count):
        smoothed_img = cv2.bilateralFilter(smoothed_img, d_size, sigma_color, sigma_space)
    
    
    color_painting = cv2.resize(smoothed_img, (original_width, original_height), interpolation=cv2.INTER_LINEAR)
    
    return color_painting

In [3]:
def combine_with_sketch(color_painting, sketch_mask):
    """
    "we start with a black background and copy the 'painting' pixels that aren't edges".
    we create a new black image. Then, look at the mask. Wherever the mask is white (meaning "not an edge"), 
    copy the pixel from your color_painting over to the new black image
    """
    
    if cv2.mean(sketch_mask)[0] < 128:
        print("Note: The sketch mask appears to be white-on-black. Inverting it for the overlay...")
        sketch_mask = cv2.bitwise_not(sketch_mask)

    
    final_output = cv2.bitwise_and(color_painting, color_painting, mask=sketch_mask)
    
    return final_output

In [4]:
def main():
    
    INPUT_IMAGE_PATH = "orgImg.png"  
    SKETCH_MASK_PATH = "sketch_mask.png" 

    
    DOWNSAMPLE_FACTOR = 0.5
    
    REPETITION_COUNT = 5  
    D_SIZE = 9            
    SIGMA_COLOR = 10      # (larger = more blending)
    SIGMA_SPACE = 10     

    
    print("Loading images...")
    original_image = cv2.imread(INPUT_IMAGE_PATH)
    if original_image is None:
        print(f"Error: Could not load image from '{INPUT_IMAGE_PATH}'")
        return

    # Load the sketch mask as grayscale
    sketch_mask = cv2.imread(SKETCH_MASK_PATH, cv2.IMREAD_GRAYSCALE)
    if sketch_mask is None:
        print(f"Error: Could not load sketch mask from '{SKETCH_MASK_PATH}'")
        
        #return

    
    print("Applying bilateral filter...")
    color_painting = create_color_painting(
        original_image,
        d_size=D_SIZE,
        sigma_color=SIGMA_COLOR,
        sigma_space=SIGMA_SPACE,
        repetition_count=REPETITION_COUNT,
        downsample_factor=DOWNSAMPLE_FACTOR
    )

   
    print("Combining painting and sketch...")
    #final_cartoon_image = combine_with_sketch(color_painting, sketch_mask)

    
    print("Done! Displaying results.")
    cv2.imshow("1. Original Image", original_image)
    cv2.imshow("2. Color Painting", color_painting)
    #cv2.imshow("3. Sketch Mask (Person 2's Part)", sketch_mask)
    #cv2.imshow("4. Final Cartoon Output", final_cartoon_image)

    
    #cv2.imwrite("final_cartoon_image.jpg", final_cartoon_image)
    print("\nFinal image saved as 'final_cartoon_image.jpg'.")
    
    
    cv2.waitKey(0)
    cv2.destroyAllWindows()


In [5]:
main()

Loading images...
Error: Could not load sketch mask from 'sketch_mask.png'
Applying bilateral filter...
Combining painting and sketch...
Done! Displaying results.

Final image saved as 'final_cartoon_image.jpg'.
